# Protocol D, fixed-majority sweep: imports and config hash

In [1]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("KAGGLE_KEY")
secret_value_1 = user_secrets.get_secret("KAGGLE_USERNAME")

In [2]:
import sys
import os
import glob
import shutil
import warnings
import numpy as np
import pandas as pd
import torch

warnings.filterwarnings(
    "ignore",
    message="The behavior of DataFrame concatenation with empty or all-NA entries is deprecated",
    category=FutureWarning,
)

try:
    from kaggle_secrets import UserSecretsClient
    _sec = UserSecretsClient()
    os.environ["KAGGLE_USERNAME"] = _sec.get_secret("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = _sec.get_secret("KAGGLE_KEY")
    print("kaggle credentials loaded from secrets")
except Exception as exc:
    print("kaggle secrets not loaded:", exc)

try:
    import core
except ModuleNotFoundError:
    hits = glob.glob("/kaggle/input/**/core.py", recursive=True)
    if not hits:
        raise ModuleNotFoundError("core.py not found.")
    sys.path.insert(0, os.path.dirname(hits[0]))
    import core

print("import-time CONFIG_HASH:", core.CONFIG_HASH)

kaggle credentials loaded from secrets
import-time CONFIG_HASH: 03fa342b535795c1


# GPU guard

In [3]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA device. Protocol D trains ResNet-18 across 45 new runs; "
        "a CPU kernel is infeasible. Switch the Kaggle accelerator to GPU (P100) and rerun. "
        "Notebook 00 can run on CPU, notebook 04 cannot."
    )
print("CUDA available:", torch.cuda.get_device_name(0))

CUDA available: Tesla T4


# Stage the frozen config, refresh, then load data

In [4]:
pulled_ok = False
if core.DATAWISE_RESULTS_SLUG is not None:
    try:
        core.pull_results_store()
        pulled_ok = True
        print("pulled results store (latest) from", core.DATAWISE_RESULTS_SLUG)
    except Exception as exc:
        print("PULL FAILED:", exc)

if not pulled_ok:
    raise RuntimeError(
        "pull failed. This session would otherwise push a fresh store over the "
        "existing one and clobber prior protocols. Fix the pull and rerun. Do not "
        "let the sweep or push proceed without the prior ledger in place."
    )

if not os.path.exists(core.PATHS["ledger"]):
    raise RuntimeError(
        f"pull reported success but no ledger at {core.PATHS['ledger']}. The store "
        "is empty or malformed. Stop and check the dataset before running the sweep."
    )

prior = pd.read_csv(core.PATHS["ledger"])
protocols = sorted(set("|".join(prior["protocol"].astype(str)).split("|")))
print(f"prior ledger rows: {len(prior)}")
print("protocols present:", protocols)

required = {"protocolA", "protocolB", "protocolC"}
missing = required - set(protocols)
if missing:
    raise RuntimeError(
        f"pulled ledger is missing {missing}. This is not the restored version 5 "
        "state. Do not run the sweep or you will push an incomplete store. Confirm "
        "the upload finished and is the latest version, then rerun."
    )
print("prior state confirmed: A, B, C present")

frozen_target = core.PATHS["frozen_config"]
if not os.path.exists(frozen_target):
    hits = glob.glob("/kaggle/input/**/frozen_config.md", recursive=True)
    if not hits:
        raise FileNotFoundError(
            "frozen_config.md not found in the working dir or any mounted dataset. "
            "Run 00_setup.ipynb and mount the results store first."
        )
    core._ensure(os.path.dirname(frozen_target))
    shutil.copy(hits[0], frozen_target)
    print("staged frozen_config.md from", hits[0])

h = core.refresh_config()
print("active CONFIG_HASH after refresh:", h)
print("core.NORM_MEAN now:", core.NORM_MEAN)
print("core.NORM_STD  now:", core.NORM_STD)

data = core.load_pcam()

Dataset URL: https://www.kaggle.com/datasets/hrushisanap/datawise-results
pulled results store (latest) from hrushisanap/datawise-results
prior ledger rows: 180
protocols present: ['protocolA', 'protocolB', 'protocolC']
prior state confirmed: A, B, C present
active CONFIG_HASH after refresh: 03fa342b535795c1
core.NORM_MEAN now: [0.700756, 0.538358, 0.691621]
core.NORM_STD  now: [0.234978, 0.277409, 0.212895]
[05:46:06] train pos/neg 131072/131072
[05:46:06] valid pos/neg (raw) 16369/16399
[05:46:06] test pos/neg (raw) 16377/16391
[05:46:07] valid balanced 32738 (16369/16369)
[05:46:07] test balanced 32754 (16377/16377)


# Build the Protocol D condition list

In [5]:
n_pos = len(data.train_pos_idx)
n_neg = len(data.train_neg_idx)
N_MAJ_FIXED = 32000
N_MIN_LIST = [1000, 2000, 4000, 8000, 16000, 32000]

assert len(N_MIN_LIST) == 6, "Protocol D has 6 conditions"

conditions = []
print(f"{'idx':>3}  {'n_min':>7}  {'n_maj':>7}  {'total N':>8}  {'r':>4}")
for i, n_min in enumerate(N_MIN_LIST):
    assert N_MAJ_FIXED % n_min == 0, f"n_min {n_min} does not divide {N_MAJ_FIXED}"
    r = N_MAJ_FIXED // n_min
    n_maj = N_MAJ_FIXED
    actual_ratio = n_maj / n_min
    assert n_min <= n_pos, f"n_min {n_min} exceeds positive pool {n_pos}"
    assert n_maj <= n_neg, f"n_maj {n_maj} exceeds negative pool {n_neg}"
    conditions.append((r, n_min, n_maj))
    print(f"{i:>3}  {n_min:>7}  {n_maj:>7}  {n_min + n_maj:>8}  {r:>4}")

print(f"\n{len(conditions)} conditions, {len(core.MODEL_NAMES)} models, "
      f"{len(core.SEEDS)} seeds -> "
      f"{len(conditions) * len(core.MODEL_NAMES) * len(core.SEEDS)} run_training calls")

idx    n_min    n_maj   total N     r
  0     1000    32000     33000    32
  1     2000    32000     34000    16
  2     4000    32000     36000     8
  3     8000    32000     40000     4
  4    16000    32000     48000     2
  5    32000    32000     64000     1

6 conditions, 3 models, 3 seeds -> 54 run_training calls


# Run the sweep (models outermost, then conditions, then seeds)

In [6]:
done = 0
for model in core.MODEL_NAMES:
    for (r, n_min, n_maj) in conditions:
        for s in core.SEEDS:
            run_id = core.run_training(
                "protocolD", model, n_min, n_maj, s, s, s, data,
                target_ratio=r, resume=True)
            done += 1
            print(f"[{done:>2}/54] {model} m={n_min} r={r} seed={s} -> {run_id}")

print("sweep calls completed:", done)

[05:46:07] logistic parameter count: 55298
[05:46:19] [protocolD][logistic][m=1000 M=32000 r=32.00][seed=21] step 250/5000 loss=0.5972
[05:46:29] [protocolD][logistic][m=1000 M=32000 r=32.00][seed=21] step 500/5000 loss=0.6193
[05:46:39] [protocolD][logistic][m=1000 M=32000 r=32.00][seed=21] step 750/5000 loss=0.9238
[05:46:48] [protocolD][logistic][m=1000 M=32000 r=32.00][seed=21] step 1000/5000 loss=0.5439
[05:46:58] [protocolD][logistic][m=1000 M=32000 r=32.00][seed=21] step 1250/5000 loss=0.9636
[05:47:08] [protocolD][logistic][m=1000 M=32000 r=32.00][seed=21] step 1500/5000 loss=0.4377
[05:47:18] [protocolD][logistic][m=1000 M=32000 r=32.00][seed=21] step 1750/5000 loss=0.4986
[05:47:27] [protocolD][logistic][m=1000 M=32000 r=32.00][seed=21] step 2000/5000 loss=0.4618
[05:47:37] [protocolD][logistic][m=1000 M=32000 r=32.00][seed=21] step 2250/5000 loss=0.3961
[05:47:47] [protocolD][logistic][m=1000 M=32000 r=32.00][seed=21] step 2500/5000 loss=0.3126
[05:47:57] [protocolD][logisti

# Session summary from the ledger (no test metrics)

In [7]:
if not os.path.exists(core.PATHS["ledger"]):
    raise FileNotFoundError("no ledger yet; run the sweep cell first")

ledger = pd.read_csv(core.PATHS["ledger"])
mask = ledger["protocol"].astype(str).str.contains("protocolD")
rows = ledger[mask]

order = {"failed": 0, "success": 1}
rows = rows.assign(_rank=rows["status"].map(order).fillna(-1))
dedup = rows.sort_values("_rank").drop_duplicates("run_id", keep="last").drop(columns="_rank")
n_success = int((dedup["status"] == "success").sum())
n_failed = int((dedup["status"] == "failed").sum())
expected = len(conditions) * len(core.MODEL_NAMES) * len(core.SEEDS)

print(f"Protocol D: {n_success} success, {n_failed} failed, "
      f"{expected - n_success} pending of {expected}")

failed_rows = dedup[dedup["status"] == "failed"]
if len(failed_rows):
    print("\nfailed runs:")
    for _, r in failed_rows.iterrows():
        print(f"  {r['run_id']}  (model={r['model']} "
              f"n_min={r['n_min']} n_maj={r['n_maj']} seed={r['data_seed']})")

core._ensure(core.PATHS["summaries"])
summary_path = os.path.join(core.PATHS["summaries"], "04_protocolD.md")
lines = [
    "# Notebook 04: Protocol D (fixed majority) session summary",
    "",
    f"Active config hash: {core.CONFIG_HASH}",
    "",
    f"- expected unique runs: {expected} (6 conditions x 3 models x 3 seeds)",
    f"- success: {n_success}",
    f"- failed: {n_failed}",
    f"- pending: {expected - n_success}",
    "- the n_min=32000 point (32000, 32000) is shared with Protocols A and B, so of the 54 "
    "expected unique runs 9 are reuses (3 models x 3 seeds) and this protocol adds 45 new runs.",
    "",
    "## Failed runs",
]
if len(failed_rows):
    for _, r in failed_rows.iterrows():
        lines.append(f"- {r['run_id']} (model={r['model']} "
                     f"n_min={r['n_min']} n_maj={r['n_maj']} seed={r['data_seed']})")
else:
    lines.append("- none")
lines.append("")
with open(summary_path, "w") as f:
    f.write("\n".join(lines))
print("\nwrote", summary_path)

Protocol D: 54 success, 0 failed, 0 pending of 54

wrote /kaggle/working/summaries/04_protocolD.md


# Persistence

In [8]:
if core.DATAWISE_RESULTS_SLUG is not None:
    core.push_results_store()
    print("pushed results store to", core.DATAWISE_RESULTS_SLUG)
else:
    print("DATAWISE_RESULTS_SLUG is unset; persistence skipped this session.")
    print("WARNING: without it, the ledger and logits will not survive session end.")

Starting upload for file random_permutations.zip


100%|██████████| 2.25M/2.25M [00:00<00:00, 4.66MB/s]


Upload successful: random_permutations.zip (2MB)
Starting upload for file frozen_config.md


100%|██████████| 431/431 [00:00<00:00, 984B/s]


Upload successful: frozen_config.md (431B)
Starting upload for file logs.zip


...

Upload successful: summaries.zip (2KB)
Starting upload for file balanced_eval.zip


100%|██████████| 139k/139k [00:00<00:00, 337kB/s]


Upload successful: balanced_eval.zip (139KB)
Starting upload for file condition_registry.csv


100%|██████████| 911/911 [00:00<00:00, 2.20kB/s]


Upload successful: condition_registry.csv (911B)
pushed results store to hrushisanap/datawise-results
